# Feature Map 크기 계산 및 실습 


$$
H_{out}=\left\lfloor \frac{H_{in}+2P_h-D_h,(K_h-1)-1}{S_h}+1 \right\rfloor
$$

$$
W_{out}=\left\lfloor \frac{W_{in}+2P_w-D_w,(K_w-1)-1}{S_w}+1 \right\rfloor
$$


$ (H_{in}, W_{in}) $: 입력 크기  
$ (K_h, K_w) $: 커널 크기  
$ (S_h, S_w) $: 스트라이드  
$ (P_h, P_w) $: 패딩  
$ (D_h, D_w) $: dilation(기본 1)  


In [27]:
import torch

# 원본 데이터: (H, W, C) 형태로 입력
raw_input_tensor = torch.tensor(
    [
        [[1, 0, 1],
         [2, 1, 0],
         [0, 1, 2],
         [2, 0, 0],
         [1, 1, 1]],

        [[0, 0, 0],
         [1, 2, 0],
         [1, 1, 0],
         [0, 1, 1],
         [0, 0, 1]],

        [[1, 0, 1],
         [0, 0, 0],
         [2, 2, 2],
         [0, 0, 1],
         [1, 1, 2]],

        [[0, 0, 1],
         [1, 0, 0],
         [1, 1, 1],
         [2, 1, 0],
         [0, 0, 0]],

        [[1, 1, 0],
         [0, 0, 0],
         [1, 2, 1],
         [0, 2, 2],
         [0, 0, 0]]
    ],
    dtype=torch.float32
)

# 원본 필터 데이터: (kH, kW, C)
raw_filter_tensor = torch.tensor(
    [
        [[1, 1, 2],
         [0, 2, 0]],

        [[0, 0, 0],
         [1, 1, 0]]
    ],
    dtype=torch.float32
)

# Conv2d 입력 형태 (N, C, H, W)로 변환
input_tensor = raw_input_tensor.permute(2, 0, 1).unsqueeze(0)

# Conv2d 가중치 형태 (out_channels, in_channels, kH, kW)로 변환
filter_tensor = raw_filter_tensor.permute(2, 0, 1).unsqueeze(0)

print("raw_input_tensor shape (H, W, C):", raw_input_tensor.shape)
print("input_tensor shape (N, C, H, W):", input_tensor.shape)
print("raw_filter_tensor shape (kH, kW, C):", raw_filter_tensor.shape)
print("filter_tensor shape (out, in, kH, kW):", filter_tensor.shape)

raw_input_tensor shape (H, W, C): torch.Size([5, 5, 3])
input_tensor shape (N, C, H, W): torch.Size([1, 3, 5, 5])
raw_filter_tensor shape (kH, kW, C): torch.Size([2, 2, 3])
filter_tensor shape (out, in, kH, kW): torch.Size([1, 3, 2, 2])


In [28]:
import torch.nn as nn

conv_layer = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=2, bias=True)

with torch.no_grad():
    conv_layer.weight.copy_(filter_tensor)
    conv_layer.bias.zero_()

output_tensor = conv_layer(input_tensor)

print("Output shape after convolution:", output_tensor.shape)
print("Output tensor after convolution:\n", output_tensor)

Output shape after convolution: torch.Size([1, 1, 4, 4])
Output tensor after convolution:
 tensor([[[[ 8.,  7.,  6.,  4.],
          [ 4.,  9.,  4.,  5.],
          [ 4.,  6., 11.,  4.],
          [ 2.,  6.,  8.,  3.]]]], grad_fn=<ConvolutionBackward0>)


## 💡 피처맵 크기 계산 공식

정방형(가로세로 길이가 같은) 이미지와 필터를 기준으로, 결과 피처맵의 한 변의 길이($O$)를 구하는 공식은 다음과 같습니다.

$$O = \lfloor \frac{I + 2P - F}{S} \rfloor + 1$$

* **$I$ (Input):** 입력 이미지의 크기
* **$P$ (Padding):** 패딩의 크기 (양쪽에 붙으므로 공식에서는 $2$를 곱해줍니다)
* **$F$ (Filter):** 필터(커널)의 크기
* **$S$ (Stride):** 스트라이드(이동 보폭) 값
* **$\lfloor \cdot \rfloor$ (내림 연산):** 계산 결과가 소수점으로 나오면 소수점 이하는 버린다는 뜻입니다.

---

## 🧮 질문하신 숫자로 직접 계산해 보기

질문해주신 조건들을 위 공식의 알파벳에 그대로 대입만 하면 됩니다.

* $I = 12$
* $F = 6$
* $S = 2$
* $P = 1$

**1단계: 입력 크기에 패딩 추가하기 ($I + 2P$)**
양쪽에 패딩이 1씩 붙으므로, 실제 입력 데이터의 길이는 $12 + (2 \times 1) = 14$가 됩니다.

**2단계: 필터 크기 빼기 ($I + 2P - F$)**
확장된 14의 길이에서 필터 크기 6을 뺍니다.
$14 - 6 = 8$

**3단계: 스트라이드로 나누기 ($/ S$)**
필터가 2칸씩 이동하므로, 앞서 나온 8을 2로 나눕니다.
$8 / 2 = 4$

**4단계: 기본값 1 더하기 ($+ 1$)**
마지막으로 필터가 처음 시작하는 위치 1자리를 더해줍니다.
$4 + 1 = 5$

결과적으로 가로 5, 세로 5의 크기가 도출되어 최종 피처맵의 크기는 **5x5**가 됩니다. 앞으로는 아무리 숫자가 커지거나 복잡해져도 이 공식에 값만 쏙쏙 대입하시면 아주 쉽게 결과를 얻으실 수 있습니다.